## Exercises on Deep Q-Networks and their Refinements

These paper-and-pencil exercises reinforce Chapter 08: the two targets of DQN vs. Double DQN, why a replay buffer decorrelates data, prioritised experience replay (priorities, sampling probabilities, importance-sampling weights), the dueling aggregation and its identifiability constraint, and Polyak (soft) target updates. Notation: $\theta$ = online weights, $\theta^-$ = target-network weights, $\delta$ = TD error, $\gamma$ = discount.

### Exercise 8.1 — DQN vs. Double DQN targets

For a sampled transition with reward $r=1$ and $\gamma=0.9$, the next state $s'$ has action-values

$\displaystyle Q(s',\cdot;\theta)=(1.0,\ 3.0)\ \text{(online)}, \qquad Q(s',\cdot;\theta^-)=(2.5,\ 2.0)\ \text{(target)}.$

Compute the **DQN** target and the **Double DQN** target, and explain why DDQN reduces overestimation.

**Step 1 — DQN target.** DQN takes the **max over the target network** (selection *and* evaluation by $\theta^-$):

$\displaystyle y_{\text{DQN}} = r + \gamma\max_{a'}Q(s',a';\theta^-) = 1 + 0.9\max(2.5,2.0) = 1 + 0.9(2.5) = 3.25.$

**Step 2 — Double DQN target.** DDQN **selects** the action with the online network and **evaluates** it with the target network:

$\displaystyle a^* = \arg\max_{a'}Q(s',a';\theta) = a_2 \ (\text{online value }3.0),$
$\displaystyle y_{\text{DDQN}} = r + \gamma\,Q(s',a^*;\theta^-) = 1 + 0.9\,Q(s',a_2;\theta^-) = 1 + 0.9(2.0) = 2.8.$

**Step 3 — Why lower.** The online argmax is $a_2$, but the *target* network rates $a_2$ at only $2.0$ (not the $2.5$ that DQN's own max would grab from $a_1$). By forcing the evaluation to come from an independent network, DDQN avoids automatically pairing "the action that looks best" with "the value that looks biggest", so $y_{\text{DDQN}}=2.8 < 3.25 = y_{\text{DQN}}$.

**Key concept**

DQN's single max both *selects* and *evaluates* with the same (noisy) estimates, producing the maximisation bias of Chapter 05 — now in a deep setting. DDQN decouples the two using the target network the algorithm already maintains, at no extra cost.

### Exercise 8.2 — Replay buffer and decorrelation

A replay buffer holds $N=100$ transitions; each optimisation step samples a mini-batch of $m=10$ **without replacement**, uniformly at random.

1. What is the probability that a specific transition is included in a given mini-batch?
2. Two *temporally consecutive* transitions (collected at steps $t$ and $t{+}1$) are strongly correlated. What is the probability that **both** land in the same mini-batch?
3. Interpret why this addresses the IID-violation problem of online updates.

**Step 1 — Inclusion probability.** With uniform sampling without replacement, each transition is equally likely to be among the $m$ chosen out of $N$:

$\displaystyle \Pr(\text{a specific transition in the batch}) = \frac{m}{N} = \frac{10}{100} = 0.1.$

**Step 2 — Both correlated samples together.** The first of the pair is included with prob. $m/N$; given that, the second occupies one of the remaining $m-1$ of the remaining $N-1$ slots:

$\displaystyle \Pr(\text{both}) = \frac{m}{N}\cdot\frac{m-1}{N-1} = \frac{10}{100}\cdot\frac{9}{99} = 0.0091\ (\approx 0.9\%).$

**Step 3 — Interpretation.** Online updates use consecutive, highly correlated transitions every step, badly violating the IID assumption of stochastic-gradient training. Sampling from a large buffer makes it *unlikely* that adjacent, correlated transitions are learned together (here under $1\%$), so each mini-batch mixes experiences from many different times and policies — approximately IID.

**Key concept**

Experience replay converts a correlated online stream into a near-IID dataset by **breaking temporal adjacency**, which is what lets standard gradient-based optimisation remain stable. It also lets each transition be reused many times, improving sample efficiency.

### Exercise 8.3 — Prioritised Experience Replay

Four transitions in the buffer have TD errors $\delta=(2.0,\,1.0,\,0.5,\,0.0)$. Prioritised replay uses priority $p_i=|\delta_i|+\epsilon$ with $\epsilon=0.5$, sampling exponent $\alpha=1$, and importance-sampling exponent $\beta=0.5$ (buffer size $N=4$).

1. Compute the priorities and the sampling probabilities $P(i)=p_i^{\alpha}/\sum_k p_k^{\alpha}$.
2. Compute the importance-sampling weights $w_i=\big(\tfrac1N\tfrac1{P(i)}\big)^{\beta}$, normalised so the largest is $1$.
3. Explain the role of $\epsilon$ and of the IS weights.

**Step 1 — Priorities and probabilities.** $p_i=|\delta_i|+0.5 = (2.5,\,1.5,\,1,\,0.5)$, sum $=5.5$. With $\alpha=1$:

$\displaystyle P = \Big(0.4545,\ 0.2727,\ 0.1818,\ 0.0909\Big).$

**Step 2 — Importance-sampling weights.** $w_i=\big(N\,P(i)\big)^{-\beta}$ with $N=4,\ \beta=0.5$:

$\displaystyle w = (0.7416,\,0.9574,\,1.1726,\,1.6583)\ \xrightarrow{\div\max} \ \tilde w = (0.4472,\,0.5774,\,0.7071,\,1).$

**Step 3 — Interpretation.** The constant $\epsilon$ keeps the zero-TD-error transition (transition 4) at a **non-zero** priority ($P=0.0909>0$), so it can still be replayed — without it, a transition that once had $\delta=0$ would never be sampled again. The IS weights **counteract the sampling bias**: the most-sampled (highest-priority) transition 1 gets the *smallest* weight $0.4472$, while the least-sampled transition 4 gets weight $1$, so that frequently-replayed transitions do not dominate the gradient.

**Key concept**

PER replays *surprising* (high-$|\delta|$) transitions more often to learn faster, but this skews the data distribution; importance-sampling weights re-balance the updates so the expected gradient stays (approximately) correct. $\alpha$ tunes how greedy the prioritisation is; $\beta$ tunes how fully the bias is corrected.

### Exercise 8.4 — Dueling architecture and identifiability

A dueling network outputs a state-value $V(s)=5$ and raw advantages $A(s,\cdot)=(1,\,-1,\,2)$ for three actions. It combines them with the **mean-subtracted** aggregation

$\displaystyle Q(s,a) = V(s) + \Big(A(s,a) - \tfrac{1}{|\mathcal{A}|}\sum_{a'}A(s,a')\Big).$

1. Compute $Q(s,\cdot)$ and verify that $V(s)=\tfrac1{|\mathcal{A}|}\sum_a Q(s,a)$.
2. Show that the *naive* aggregation $Q=V+A$ is **unidentifiable**, and explain how the mean-subtraction fixes it.

**Step 1 — Aggregate.** Mean advantage $\bar A=\tfrac{1+(-1)+2}{3}=0.6667$. Then

$\displaystyle Q(s,\cdot) = 5 + \big[(1,-1,2)-0.6667\big] = (5.3333, 3.3333, 6.3333).$

Check: $\tfrac13\sum_a Q(s,a) = \tfrac{5.3333+3.3333+6.3333}{3} = 5 = V(s)$. ✓ The mean-subtraction forces the value head to equal the mean action-value.

**Step 2 — Unidentifiability of $Q=V+A$.** The same $Q(s,\cdot)=(5.3333, 3.3333, 6.3333)$ can be produced by infinitely many $(V,A)$ splits, e.g.

- $V=5,\ A=(0.3333,-1.6667,1.3333)$ (mean $0$), or
- $V=0,\ A=(5.3333,3.3333,6.3333)$ (mean $5$),

since only the *sum* $V+A$ enters $Q$. Given only $Q$, we cannot recover $V$ and $A$ separately — the decomposition is not unique.

**Step 3 — The fix.** Imposing $\sum_a A(s,a)=0$ (equivalently, subtracting the mean advantage) removes this freedom: it pins $V(s)=\tfrac1{|\mathcal{A}|}\sum_a Q(s,a)$ and $A(s,a)=Q(s,a)-V(s)$, a *unique* decomposition. This is why the aggregation subtracts $\bar A$ rather than using $V+A$ directly.

**Key concept**

The dueling network separately estimates "how good is this state" ($V$) and "how much better is each action" ($A$), sharing features so that learning about one action informs the state value. The mean-subtraction is not cosmetic — it is what makes the two heads *identifiable* and stably trainable.

### Exercise 8.5 — Polyak (soft) target updates

A target network is updated softly by $\theta^- \leftarrow \tau\,\theta + (1-\tau)\,\theta^-$ with mixing factor $\tau=0.1$. For a single scalar weight, suppose the online weight is fixed at $\theta=10$ and the target starts at $\theta^-_0=0$.

1. Compute $\theta^-$ after three soft updates.
2. Give the closed form $\theta^-_k$ and contrast soft updates with a hard periodic copy.

**Step 1 — Iterate** $\theta^-\leftarrow 0.1(10)+0.9\,\theta^-$:

$\displaystyle \theta^-_1 = 0.1(10)+0.9(0) = 1,\quad \theta^-_2 = 1+0.9(1) = 1.9,\quad \theta^-_3 = 1+0.9(1.9) = 2.71.$

**Step 2 — Closed form.** Unrolling with constant online weight $\theta=10$:

$\displaystyle \theta^-_k = \theta\big(1-(1-\tau)^k\big) = 10\big(1-0.9^{k}\big),$

giving $10(1-0.9)=1,\ 10(1-0.81)=1.9,\ 10(1-0.729)=2.71$ — matching Step 1. The target **exponentially approaches** the online weight, always lagging.

**Step 3 — Soft vs. hard.** A *hard* update copies $\theta$ into $\theta^-$ every $C$ steps: the target is perfectly stable between copies but jumps discontinuously. A *soft* (Polyak) update moves the target a little ($\tau$ small) every step: it lags smoothly, avoiding the large discrete jump while still changing slowly enough to stabilise the bootstrapped target.

**Key concept**

The target network trades learning speed for stability; Polyak averaging with small $\tau$ makes that trade **smooth and continuous** instead of abrupt. (Note the ordering: $\tau$ multiplies the *online* weights — a small $\tau$ means a *small* step toward the online network, i.e. a slow-moving target.)